# Task 5 Build a Traffic Recommendation System

I woudl build a recommendation system for choosing lower-traffic travel windows on the corridor represented by the dataset.

The system would compare predicted traffic volume across candidate hours using time and weather inputs, then provide a plain-language recommendation.

The dataset represents one corridor, so the system does not recommend alternative routes.

In [1]:
# Configure logging and load the saved regression artifacts.


import logging
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

PART3_DIR = Path.cwd().parent
REPO_DIR = PART3_DIR.parent

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(
            PART3_DIR / "part3_traffic_recommendation.log",
            encoding="utf-8"
        )
    ],
    force=True
)






logger = logging.getLogger(__name__)

logger.info("Traffic recommendation notebook has started.")

recommendation_model = joblib.load(
    PART3_DIR / "models" / "random_forest_regression_v1.joblib"
)

recommendation_config = joblib.load(
    PART3_DIR / "models" / "preparation_config_v1.joblib"
)

logger.info(
    "Loaded Random Forest regression v1 and its preparation settings."
)



display(pd.DataFrame([{
    "model": "Random Forest regression",
    "version": "v1",
    "target": "traffic_volume",
    "input_features": len(
        recommendation_config["feature_columns"]
    ),
    "scope": "Travel windows on one corridor"
}]))



2026-10-11 10:51:18,238 | INFO | __main__ | Traffic recommendation notebook has started.
2026-10-11 10:51:23,017 | INFO | __main__ | Loaded Random Forest regression v1 and its preparation settings.


,model,version,target,input_features,scope
0,Random Forest regression,v1,traffic_volume,17,Travel windows on one corridor


## Validate inputs and prepare candidate travel hours

Temperature is supplied in Kelvin, precipitation in millimetres and cloud cover as a percentage.

I wouldvalidate the inputs and recreate the same features used during model training. 

Each row represents the conditions assumed for one candidate hour

In [2]:
# Validate candidate conditions and recreate the model features.


weather_encoder = (
    recommendation_model.named_steps["preprocessor"]
    .named_transformers_["weather"]
)

known_weather = set(weather_encoder.categories_[0])



def prepare_recommendation_inputs(candidates):
    prepared = candidates.copy()

    required_columns = [
        "date_time",
        "temp",
        "rain_1h",
        "snow_1h",
        "clouds_all",
        "weather_main",
        "is_holiday"
    ]

    missing_columns = [
        column for column in required_columns
        if column not in prepared.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Required columns are missing: {missing_columns}."
        )

    if prepared.empty:
        raise ValueError("At least one candidate hour is required.")

    
    prepared["date_time"] = pd.to_datetime(
        prepared["date_time"],
        errors="coerce"
    )

    if prepared["date_time"].isna().any():
        raise ValueError("Candidate timestamps must be valid.")

    if prepared["date_time"].dt.tz is not None:
        raise ValueError(
            "Use local corridor timestamps without timezone offsets."
        )

    if not prepared["date_time"].eq(
        prepared["date_time"].dt.floor("h")
    ).all():
        raise ValueError("Candidate timestamps must be on the hour.")

    if prepared["date_time"].duplicated().any():
        raise ValueError("Candidate timestamps must be unique.")

    # Require explicit, finite measurements for this recommendation.
    numeric_columns = [
        "temp", "rain_1h", "snow_1h", "clouds_all", "is_holiday"
    ]

    
    for column in numeric_columns:
        prepared[column] = pd.to_numeric(
            prepared[column], errors="coerce"
        )

    if not np.isfinite(
        prepared[numeric_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError("Numeric inputs must be finite numbers.")

    if not prepared["temp"].between(200, 330).all():
        raise ValueError(
            "Temperature must be between 200 and 330 Kelvin "
        )

    if not prepared["rain_1h"].between(0, 9000).all():
        raise ValueError("Rainfall must be between 0 and 9000 mm.")

    if (prepared["snow_1h"] < 0).any():
        raise ValueError("Snowfall must be non-negative.")

    if not prepared["clouds_all"].between(0, 100).all():
        raise ValueError("Cloud cover must be between 0 and 100.")

    if not prepared["is_holiday"].isin([0, 1]).all():
        raise ValueError("The holiday flag must be 0 or 1.")

    prepared["weather_main"] = (
        prepared["weather_main"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    if not prepared["weather_main"].isin(known_weather).all():
        raise ValueError(
            "Weather categories must match those used in training."
        )

    # Recreate time and weather features.
    prepared["hour"] = prepared["date_time"].dt.hour
    prepared["day_of_week"] = prepared["date_time"].dt.dayofweek
    prepared["month"] = prepared["date_time"].dt.month
    prepared["is_weekend"] = (
        prepared["day_of_week"] >= 5
    ).astype(int)

    hour_angle = 2 * np.pi * prepared["hour"] / 24
    weekday_angle = 2 * np.pi * prepared["day_of_week"] / 7

    prepared["hour_sin"] = np.sin(hour_angle)
    prepared["hour_cos"] = np.cos(hour_angle)
    prepared["weekday_sin"] = np.sin(weekday_angle)
    prepared["weekday_cos"] = np.cos(weekday_angle)


    
    prepared["has_rain"] = (
        prepared["rain_1h"] > 0
    ).astype(int)

    prepared["has_snow"] = (
        prepared["snow_1h"] > 0
    ).astype(int)

    prepared["is_low_visibility"] = (
        prepared["weather_main"].isin(
            recommendation_config["low_visibility_weather"]
        )
    ).astype(int)

    return prepared[
        recommendation_config["feature_columns"]
    ].copy()


logger.info(
    "Recommendation input validation and feature preparation is  ready now."
)



display(pd.DataFrame({
    "accepted_weather_categories": sorted(known_weather)
}))

2026-10-11 15:51:09,925 | INFO | __main__ | Recommendation input validation and feature preparation is  ready now.


,accepted_weather_categories
0,Clear
1,Clouds
2,Drizzle
3,Fog
4,Haze
5,Mist
6,Rain
7,Smoke
8,Snow
9,Squall


## Compare candidate travel windows

I can predict traffic volume for each supplied candidate hour and rank the hours from lower to higher predicted volume. Prefernce is given to the earolier timestamp.

The recommendation woudl identify the lowest-volume hour among the supplied choices. 

I am assume=ing the same clear weather conditions across the candidate hours. 

In [4]:
#Rank supplied hours and create a plain-language recommendation.

def recommend_travel_window(candidates):
    try:
        model_inputs = prepare_recommendation_inputs(candidates)
        predictions = recommendation_model.predict(model_inputs)

        if not np.isfinite(predictions).all():
            raise ValueError("Model predictions must be finite.")

        
        ranked = candidates.copy()
        ranked["date_time"] = pd.to_datetime(ranked["date_time"])
        ranked["predicted_traffic_volume"] = predictions

        ranked = ranked.sort_values(
            ["predicted_traffic_volume", "date_time"],
            kind="stable"
        ).reset_index(drop=True)

        ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))

        best = ranked.iloc[0]
        highest = ranked["predicted_traffic_volume"].max()
        difference = highest - best["predicted_traffic_volume"]

        message = (
            f"Among the {len(ranked)} supplied candidate hours, "
            f"{best['date_time']:%d %B %Y at %H:%M} has the "
            f"lowest predicted traffic volume, approximately "
            f"{best['predicted_traffic_volume']:.0f} vehicles "
            f"during that hour. "
            f"This is approximately {difference:.0f} fewer than "
            f"the highest-volume candidate. "
            "The comparison uses yje  supplied weather and holiday "
            "assumptions; it is not live traffic information or "
            "a journey-time estimate."
        )

        logger.info(
            "Ranked %s candidate hours and prepared a recommendation.",
            len(ranked)
        )

        return ranked, message


    except (ValueError, TypeError, KeyError):
        logger.error("Travel-window recommendation failed.")
        raise


# Historical demonstration with explicit weather assumptions.
candidate_hours = pd.date_range(
    "2018-05-07 06:00:00",
    "2018-05-07 10:00:00",
    freq="h"
)



demo_candidates = pd.DataFrame({
    "date_time": candidate_hours,
    "temp": 288.15,
    "rain_1h": 0.0,
    "snow_1h": 0.0,
    "clouds_all": 0.0,
    "weather_main": "Clear",
    "is_holiday": 0
})

ranked_windows, recommendation_text = recommend_travel_window(
    demo_candidates
)

#MAke sure that recommendation is not trucnated by pands

from IPython.display import Markdown

display(
    ranked_windows[
        ["rank", "date_time", "predicted_traffic_volume"]
    ].rename(columns={
        "rank": "Rank",
        "date_time": "Candidate hour",
        "predicted_traffic_volume": "Predicted hourly traffic volume"
    }).style.hide(axis="index").format({
        "Candidate hour": lambda value: value.strftime(
            "%d %b %Y %H:%M"
        ),
        "Predicted hourly traffic volume": "{:,.0f}"
    })
)

display(Markdown(
    f"**Recommended travel window**\n\n{recommendation_text}"
))

2026-10-11 15:55:55,131 | INFO | __main__ | Ranked 5 candidate hours and prepared a recommendation.


Rank,Candidate hour,Predicted hourly traffic volume
1,07 May 2018 10:00,"4,204"
2,07 May 2018 09:00,"4,956"
3,07 May 2018 07:00,"5,475"
4,07 May 2018 06:00,"5,692"
5,07 May 2018 08:00,"5,871"


**Recommended travel window**

Among the 5 supplied candidate hours, 07 May 2018 at 10:00 has the lowest predicted traffic volume, approximately 4204 vehicles during that hour. This is approximately 1667 fewer than the highest-volume candidate. The comparison uses yje  supplied weather and holiday assumptions; it is not live traffic information or a journey-time estimate.

In [6]:
#Check ranking correctness and independence from input order.

assert len(ranked_windows) == len(demo_candidates)
assert ranked_windows["predicted_traffic_volume"].is_monotonic_increasing

shuffled_candidates = demo_candidates.sample(
    frac=1,
    random_state=42
)

shuffled_ranking, _ = recommend_travel_window(
    shuffled_candidates
)

pd.testing.assert_frame_equal(
    ranked_windows[["date_time", "predicted_traffic_volume"]],
    shuffled_ranking[["date_time", "predicted_traffic_volume"]]
)

# Prepare examples that can fail validation.
invalid_weather = demo_candidates.copy()
invalid_weather.loc[0, "weather_main"] = "UnknownWeather"

duplicate_time = demo_candidates.copy()
duplicate_time.loc[1, "date_time"] = duplicate_time.loc[
    0, "date_time"
]

invalid_cloud_cover = demo_candidates.copy()
invalid_cloud_cover.loc[0, "clouds_all"] = 150

validation_records = []

for case_name, invalid_input in [
    ("Unknown weather category", invalid_weather),
    ("Duplicate timestamp", duplicate_time),
    ("Cloud cover above 100", invalid_cloud_cover)
]:
    try:
        prepare_recommendation_inputs(invalid_input)
    except ValueError as error:
        validation_records.append({
            "check": case_name,
            "result": "PASS",
            "message": str(error)
        })
        logger.info(
            "Input validation check passed for %s.",
            case_name
        )
    else:
        raise AssertionError(
            f"Invalid input was accepted: {case_name}."
        )

validation_checks = pd.DataFrame(validation_records)



logger.info(
    "Ranking and input-order checks ahjve passed."
)

# Make sure that validato is not trucnated
with pd.option_context("display.max_colwidth", None):
    display(validation_checks.style.hide(axis="index"))

2026-10-11 15:59:16,573 | INFO | __main__ | Ranked 5 candidate hours and prepared a recommendation.
2026-10-11 15:59:16,583 | INFO | __main__ | Input validation check passed for Unknown weather category.
2026-10-11 15:59:16,586 | INFO | __main__ | Input validation check passed for Duplicate timestamp.
2026-10-11 15:59:16,594 | INFO | __main__ | Input validation check passed for Cloud cover above 100.
2026-10-11 15:59:16,596 | INFO | __main__ | Ranking and input-order checks ahjve passed.


check,result,message
Unknown weather category,PASS,Weather categories must match those used in training.
Duplicate timestamp,PASS,Candidate timestamps must be unique.
Cloud cover above 100,PASS,Cloud cover must be between 0 and 100.


## Save the recommendation results

I woulod, save the supplied candidate conditions, ranked travel windows, recommendation text and validation results.

These files document the historical demonstration and its weather assumptions.

In [7]:
#Save the inputs, results and validation preoofs.


results_dir = PART3_DIR / "results"
recommendation_dir = PART3_DIR / "recommendation_system"

results_dir.mkdir(parents=True, exist_ok=True)
recommendation_dir.mkdir(parents=True, exist_ok=True)

demo_candidates.to_csv(
    recommendation_dir / "example_candidate_hours.csv",
    index=False
)

ranked_windows.to_csv(
    results_dir / "recommended_travel_windows.csv",
    index=False
)

validation_checks.to_csv(
    results_dir / "recommendation_validation_checks.csv",
    index=False
)

recommendation_path = (
    results_dir / "traffic_recommendation.txt"
)



recommendation_path.write_text(
    recommendation_text + "\n",
    encoding="utf-8"
)

# Verify that the saved results retain the ranking and full text.
saved_ranking = pd.read_csv(
    results_dir / "recommended_travel_windows.csv",
    parse_dates=["date_time"]
)

np.testing.assert_allclose(
    saved_ranking["predicted_traffic_volume"],
    ranked_windows["predicted_traffic_volume"]
)

assert (
    saved_ranking["date_time"].tolist()
    == ranked_windows["date_time"].tolist()
)

assert (
    recommendation_path.read_text(encoding="utf-8").strip()
    == recommendation_text
)

logger.info(
    "Saved the recommendation inputs, ranking, text and validation results."

)


logger.info(
    "Saved recommendation results passed the read-back checks."
)



2026-10-11 16:02:21,589 | INFO | __main__ | Saved the recommendation inputs, ranking, text and validation results.
2026-10-11 16:02:21,591 | INFO | __main__ | Saved recommendation results passed the read-back checks.


In [9]:
# Export the existing function definitions from this notebook.
import ast

function_names = [
    "prepare_recommendation_inputs",
    "recommend_travel_window"
]



function_sources = {}

for cell_source in get_ipython().history_manager.input_hist_raw:
    try:
        parsed_cell = ast.parse(cell_source)
    except SyntaxError:
        continue

    for node in parsed_cell.body:
        if (
            isinstance(node, ast.FunctionDef)
            and node.name in function_names
        ):
            function_sources[node.name] = ast.get_source_segment(
                cell_source, node
            )



missing_functions = [
    name for name in function_names
    if name not in function_sources
]

if missing_functions:
    raise ValueError(
        f"Run the function-definition cells first: {missing_functions}"
    )

module_header = '''"""Recommend travel windows using supplied candidate conditions."""

import logging
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

PART3_DIR = Path(__file__).resolve().parent.parent

recommendation_model = joblib.load(
    PART3_DIR / "models" / "random_forest_regression_v1.joblib"
)

recommendation_config = joblib.load(
    PART3_DIR / "models" / "preparation_config_v1.joblib"
)

weather_encoder = (
    recommendation_model.named_steps["preprocessor"]
    .named_transformers_["weather"]
)

known_weather = set(weather_encoder.categories_[0])
'''



module_source = module_header + "\n\n" + "\n\n".join(
    function_sources[name] for name in function_names
) + "\n"


# Check syntax before saving the module.
compile(module_source, "recommender.py", "exec")

recommendation_dir = PART3_DIR / "recommendation_system"
recommendation_dir.mkdir(parents=True, exist_ok=True)

module_path = recommendation_dir / "recommender.py"
module_path.write_text(module_source, encoding="utf-8")



logger.info(
    "Saved reusable recommendation functions to recommendation_system/recommender.py."
)


2026-10-11 16:08:46,150 | INFO | __main__ | Saved reusable recommendation functions to recommendation_system/recommender.py.


### Verify the reusable recommendation module

I would load the saved module and check that its rankings and recommendation text match the notebook results.

In [10]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "traffic_recommender",
    recommendation_dir / "recommender.py"
)


recommender = importlib.util.module_from_spec(spec)
spec.loader.exec_module(recommender)

module_ranking, module_text = recommender.recommend_travel_window(
    demo_candidates
)

pd.testing.assert_frame_equal(module_ranking, ranked_windows)
assert module_text == recommendation_text

logger.info(
    "Reusable recommendation module passed the ranking and text checks."
)



2026-10-11 16:10:09,334 | INFO | traffic_recommender | Ranked 5 candidate hours and prepared a recommendation.
2026-10-11 16:10:09,338 | INFO | __main__ | Reusable recommendation module passed the ranking and text checks.


### Task 5 results and limitations

The system ranked five supplied candidate hours using the saved Random Forest regression model. For this historical example, 07 May 2018 at 10:00 had the lowest predicted hourly traffic volume, approximately 4,204 vehicles.

Input validation rejected unknown weather categories, duplicate timestamps, and cloud cover above 100. 

The recommendation compares travel hours on one corridor using supplied weather and holiday assumptions. It does not estimate journey time, compare routes, or provide live traffic information. 